# INTHON 2026 · Final model on Colab

Produces the team's final model on a T4. Run from the top with **Runtime → Change runtime type → T4 GPU**.

Before starting, copy the repository to `MyDrive/starterkit`, without `cuda/`, and the data to
`MyDrive/data/Train.zip`. `SMOKE_TEST` starts as `True` to check that everything works. **Set it to
`False` in section 2 for the final run,** and don't submit a smoke-test ZIP.

Everything saved for inspection goes to `starterkit/colab_runs/<experiment name>/`: code, settings,
logs, checkpoint, the negatives used in every epoch, and the ZIP.

Scores here use our own generated negatives, so they aren't leaderboard estimates.

## 1. Connect Google Drive
Allow access in the Google account connection window. If you use a shared folder,
**add a shortcut to My Drive**, then enter that path below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. Paths and training settings
Set `DRIVE_STARTERKIT` and `TRAIN_SOURCE` to match your Drive paths.
`TRAIN_SOURCE` is **the Train folder or a ZIP file**.

The default is the full Train set, up to 20 epochs. To check first that it works,
change to `SMOKE_TEST = True`, which runs only 3 epochs on at most 12 episodes.
The experiment name is generated automatically, and rerunning this cell starts a new experiment.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid

DRIVE_STARTERKIT = Path("/content/drive/MyDrive/starterkit")
TRAIN_SOURCE = Path("/content/drive/MyDrive/data/Train.zip")  # Or a Train folder
SMOKE_TEST = True  # Set to False for the final run
EPOCHS = 60
MODEL = "temporal"
TRAIN_NEGATIVES = "reorder,swapped_action,similar_episode,random,lexical,word_swap"
REVERSE_WEIGHT = 0.1
FLIP_PROBABILITY = 0.5
IMAGE_BATCH_SIZE = 8  # T4 default. Reduce to 4 or 2 if GPU memory runs out.
TEXT_BATCH_SIZE = 16
DOWNLOAD_ZIP_TO_COMPUTER = False  # Also runs a browser download in the last cell

RUN_NAME = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
if SMOKE_TEST:
    RUN_NAME = "smoke_" + RUN_NAME
DRIVE_RUN = DRIVE_STARTERKIT / "colab_runs" / RUN_NAME
LOCAL_ROOT = Path("/content/inthon") / RUN_NAME
LOCAL_KIT = LOCAL_ROOT / "starterkit"
CHECKPOINT_DIR = DRIVE_RUN / "training"
EXPORT_DIR = LOCAL_ROOT / "submission"
LOCAL_ZIP = LOCAL_ROOT / f"{RUN_NAME}_submission.zip"

required = ["config.json", "train.py", "evaluate.py", "export.py", "validate.py",
            "package.py", "submission.py", "baseline_model.py", "common.py", "dataset.py",
            "download_models.py", "team/negatives.py", "team/lexical.py", "team/word_swap.py", "team/train_negatives.py",
            "team/train_ranker.py", "team/ranker_model.py", "team/rank_eval.py",
            "team/submission_ranker.py", "team/export_ranker.py"]
missing = [name for name in required if not (DRIVE_STARTERKIT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Check the starterkit path or files: {DRIVE_STARTERKIT} / {missing}")
if not TRAIN_SOURCE.exists():
    raise FileNotFoundError(f"Train data not found. Fix TRAIN_SOURCE: {TRAIN_SOURCE}")
if not isinstance(EPOCHS, int) or EPOCHS < 1 or min(IMAGE_BATCH_SIZE, TEXT_BATCH_SIZE) < 1:
    raise ValueError("Epochs and batch sizes must be positive integers.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Change the Colab runtime type to T4 GPU, then run from the beginning.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                "--format=csv,noheader"], check=True)
DRIVE_RUN.mkdir(parents=True, exist_ok=False)
LOCAL_KIT.mkdir(parents=True, exist_ok=False)
print("Experiment:", RUN_NAME)
print("Drive save location:", DRIVE_RUN)


## 3. Install the competition's runtime environment
Scoring environment as of **2026-10-31**. Installs Python **3.12.3**, PyTorch **2.8.0 / CUDA 12.8**,
Transformers **4.57.6** and the supporting packages provided for the competition.

To reduce conflicts with Colab's default packages, this creates a separate Python environment in `/content`,
and all later training commands run with that environment's `PYTHON`. **No runtime restart is needed.**
Installation needs several GB of downloads and a few minutes.

This notebook uses the version list below. Do not additionally install the starter kit's existing
`requirements.txt`. When running separate code, you must also use `run_python(...)` or `run_script(...)`
for the same environment to apply. `import torch` in a normal cell uses Colab's default environment.

In [ ]:
PYTHON_VERSION = "3.12.3"
ENV_DIR = Path("/content/inthon-python312-cu128")
TOOL_DIR = Path("/content/inthon-colab-tools")
UV = TOOL_DIR / "bin/uv"
if not UV.is_file():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
                    "--target", str(TOOL_DIR), "uv==0.13.4"], check=True)
if not (ENV_DIR / "bin/python").is_file():
    subprocess.run([str(UV), "python", "install", PYTHON_VERSION], check=True)
    subprocess.run([str(UV), "venv", "--python", PYTHON_VERSION,
                    "--seed", str(ENV_DIR)], check=True)
PYTHON = ENV_DIR / "bin/python"
subprocess.run([str(PYTHON), "-c",
                "import sys; assert sys.version_info[:3] == (3, 12, 3), sys.version"], check=True)

TORCH_PINS = ["torch==2.8.0+cu128", "torchvision==0.23.0+cu128", "torchaudio==2.8.0+cu128"]
PACKAGE_PINS = [
    "transformers==4.57.6", "numpy==2.5.4", "Pillow==12.3.0", "safetensors==0.8.0",
    "huggingface-hub==0.36.2", "tokenizers==0.22.2",
    "sentence-transformers==5.4.0", "einops==0.8.2", "timm==1.0.30",
    "scipy==1.18.1", "scikit-learn==1.9.1", "opencv-python-headless==4.14.0.94",
    "peft==0.18.1", "accelerate==1.15.0", "bitsandbytes==0.50.2",
    "cloudpickle==3.1.2", "joblib==1.6.0", "narwhals==2.28.0", "threadpoolctl==3.7.0",
]
PINS_PATH = DRIVE_RUN / "requirements-colab.txt"
PINS_PATH.write_text("\n".join(TORCH_PINS + PACKAGE_PINS) + "\n", encoding="utf-8")
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
                "--index-url", "https://download.pytorch.org/whl/cu128", *TORCH_PINS], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "--no-cache-dir",
                "-r", str(PINS_PATH)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)

ENV = os.environ.copy()
ENV.pop("PYTHONPATH", None)
ENV.pop("PYTHONHOME", None)
ENV.update({"HF_HOME": "/content/inthon-hf-cache", "HF_HUB_ENABLE_HF_TRANSFER": "0",
            "TOKENIZERS_PARALLELISM": "false", "PYTHONUNBUFFERED": "1",
            "PYTHONNOUSERSITE": "1", "OMP_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2"})
for name in ("HF_HUB_OFFLINE", "TRANSFORMERS_OFFLINE"):
    ENV.pop(name, None)

def run_command(args, *, log_name=None, extra_env=None, cwd=None):
    command = [str(arg) for arg in args]
    log = (DRIVE_RUN / log_name).open("a", encoding="utf-8") if log_name else None
    try:
        with subprocess.Popen(command, cwd=cwd or LOCAL_KIT,
                              env={**ENV, **(extra_env or {})}, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    if log:
                        log.write(line)
                        log.flush()
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
        if returncode:
            raise subprocess.CalledProcessError(returncode, command)
    finally:
        if log:
            log.close()

def run_python(source, *args, **kwargs):
    run_command([PYTHON, "-u", "-c", source, *args], **kwargs)

def run_script(script, *args, **kwargs):
    run_command([PYTHON, "-u", LOCAL_KIT / script, *args], **kwargs)

freeze = subprocess.check_output([str(PYTHON), "-m", "pip", "freeze", "--all"], text=True, env=ENV)
(DRIVE_RUN / "requirements-installed.txt").write_text(freeze, encoding="utf-8")
print("Runtime environment ready:", PYTHON)


## 4. Check the GPU and major packages
Checks actual CUDA operations, package imports and bitsandbytes 4-bit operations.
Results are kept in `environment.json` on Drive. Colab's GPU and driver differ from those of the
scoring server, so this does not mean run times or numeric results will be exactly the same.

In [ ]:
run_python(r"""
import importlib, importlib.metadata as metadata, json, platform, sys
from pathlib import Path
import torch
for module in ("torchvision", "torchaudio", "transformers", "numpy", "PIL", "safetensors",
               "sentence_transformers", "einops", "timm", "scipy", "sklearn", "cv2",
               "peft", "accelerate", "bitsandbytes"):
    importlib.import_module(module)
pins = Path(sys.argv[1]).read_text().splitlines()
versions = {}
for pin in pins:
    name, expected = pin.split("==")
    versions[name] = metadata.version(name)
    assert versions[name] == expected, (name, versions[name], expected)
assert torch.cuda.is_available(), "CUDA is not available. Check the GPU runtime."
device = torch.device("cuda:0")
x = torch.randn(32, 64, device=device, dtype=torch.float16)
assert torch.isfinite(x @ x.T).all()
import bitsandbytes as bnb
layer = bnb.nn.Linear4bit(64, 16, bias=False, compute_dtype=torch.float16,
                         quant_type="nf4").to(device)
with torch.inference_mode():
    assert torch.isfinite(layer(x)).all()
torch.cuda.synchronize()
report = {"python": platform.python_version(), "packages": versions,
          "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,
          "vram_gib": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
          "cuda_and_bnb_nf4_check": "passed"}
Path(sys.argv[2]).write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")
print(json.dumps(report, ensure_ascii=False, indent=2))
""", PINS_PATH, DRIVE_RUN / "environment.json", log_name="environment.log")


## 5. Copy the code and Train data to Colab's local disk
Copies to `/content` so that training does not repeatedly read many image files from Drive.
For a large dataset, uploading a ZIP rather than a folder can make preparation faster.
A copy of the training code is also kept in the experiment folder on Drive.

In [ ]:
import stat
import zipfile

ignored = shutil.ignore_patterns(".venv", ".venv-*", "__pycache__", ".cache", ".pytest_cache",
                                ".git", "cuda", "models", "outputs", "sample_submission", "colab_runs",
                                "data", "Train", "Test", "*.zip", "*.ipynb", ".ipynb_checkpoints")
shutil.copytree(DRIVE_STARTERKIT, LOCAL_KIT, dirs_exist_ok=True, ignore=ignored)
shutil.copytree(LOCAL_KIT, DRIVE_RUN / "source", dirs_exist_ok=True)

STAGED_DATA = LOCAL_ROOT / "data"
marker = LOCAL_ROOT / "data-ready.json"
source_stat = TRAIN_SOURCE.stat()
source_stamp = {"path": str(TRAIN_SOURCE), "size": source_stat.st_size,
                "mtime_ns": source_stat.st_mtime_ns}
if marker.is_file():
    saved = json.loads(marker.read_text())
    if saved["source"] != source_stamp:
        raise RuntimeError("The input data has changed. Create a new experiment in cell 2.")
    DATA_DIR = Path(saved["data_dir"])
    print("Reusing already copied data:", DATA_DIR)
else:
    if STAGED_DATA.exists():
        raise RuntimeError(f"An incomplete copy folder exists. Create a new experiment in cell 2: {STAGED_DATA}")
    free_bytes = shutil.disk_usage(LOCAL_ROOT).free
    reserve = 4 * 1024**3  # Space for encoders, export and ZIP creation
    if TRAIN_SOURCE.is_dir():
        size = sum(p.stat().st_size for p in TRAIN_SOURCE.rglob("*") if p.is_file())
        if size + reserve > free_bytes:
            raise RuntimeError("Not enough free space on Colab's local disk.")
        print(f"Starting Train folder copy: {size / 2**30:.2f} GiB", flush=True)
        shutil.copytree(TRAIN_SOURCE, STAGED_DATA / "Train")
    elif zipfile.is_zipfile(TRAIN_SOURCE):
        # Read the archive on Drive only once and extract it locally.
        with zipfile.ZipFile(TRAIN_SOURCE) as archive:
            expanded = sum(item.file_size for item in archive.infolist())
        if source_stat.st_size + expanded + reserve > free_bytes:
            raise RuntimeError("Not enough local disk space to copy and extract the ZIP.")
        local_archive = LOCAL_ROOT / "train-input.zip"
        shutil.copyfile(TRAIN_SOURCE, local_archive)
        STAGED_DATA.mkdir(parents=True)
        root = STAGED_DATA.resolve()
        with zipfile.ZipFile(local_archive) as archive:
            for item in archive.infolist():
                destination = (root / item.filename).resolve()
                if not destination.is_relative_to(root) or stat.S_ISLNK(item.external_attr >> 16):
                    raise ValueError(f"Unsupported ZIP path: {item.filename}")
            archive.extractall(root)
        local_archive.unlink()
    else:
        raise ValueError("Set TRAIN_SOURCE to the Train folder or a ZIP file.")
    candidates = [STAGED_DATA, STAGED_DATA / "Train", STAGED_DATA / "data/Train"]
    matches = [p for p in candidates if any(p.glob("episode_*/description.txt"))]
    if len(matches) != 1:
        raise ValueError("Check the Train folder structure: episode_*/description.txt and images/ are required.")
    DATA_DIR = matches[0]
    marker.write_text(json.dumps({"source": source_stamp, "data_dir": str(DATA_DIR)}, indent=2))
print("Training data:", DATA_DIR)
print("Local code:", LOCAL_KIT)


## 6. Training settings and encoder preparation
Copies `models/dinov2` and `models/mpnet` if they are on Drive; otherwise downloads the two public encoders.
Downloads are needed only in the preparation step. Later training and validation use the local models.
Also checks in advance that each Train split has at least two distinct descriptions.

In [ ]:
CONFIG_PATH = LOCAL_KIT / "config-colab.json"
config = json.loads((LOCAL_KIT / "config.json").read_text(encoding="utf-8"))
config.update({"data_dir": str(DATA_DIR), "image_model_dir": str(LOCAL_KIT / "models/dinov2"),
               "text_model_dir": str(LOCAL_KIT / "models/mpnet"),
               "output_dir": str(CHECKPOINT_DIR), "epochs": 3 if SMOKE_TEST else EPOCHS,
               "image_batch_size": IMAGE_BATCH_SIZE, "text_batch_size": TEXT_BATCH_SIZE,
               "cpu_threads": 2})
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
shutil.copyfile(CONFIG_PATH, DRIVE_RUN / "config-colab.json")
run_python(r"""
import json, random, sys
from dataset import load_episodes, split_indices
cfg = json.load(open(sys.argv[1]))
episodes = load_episodes(cfg["data_dir"])
total = len(episodes)
if sys.argv[2] == "True":
    indices = random.Random(cfg["seed"]).sample(range(total), min(12, total))
    episodes = [episodes[i] for i in sorted(indices)]
train, valid = split_indices(episodes, cfg["validation_fraction"], cfg["seed"])
for name, indices in (("train", train), ("valid", valid)):
    assert len({episodes[i].description for i in indices}) >= 2, f"{name}: at least two distinct descriptions are required."
print(f"Data check complete: total {total} / used {len(episodes)} / train {len(train)} / validation {len(valid)}")
""", CONFIG_PATH, str(SMOKE_TEST), log_name="data-check.log")

model_files = {"dinov2": ["config.json", "model.safetensors", "preprocessor_config.json"],
               "mpnet": ["config.json", "model.safetensors", "tokenizer.json", "tokenizer_config.json"]}
source_models = DRIVE_STARTERKIT / "models"
models_complete = all((source_models / name / file).is_file()
                      for name, files in model_files.items() for file in files)
model_args = ["--config", CONFIG_PATH]
if models_complete:
    model_args += ["--source-root", source_models]
    print("Copying the encoders from Drive.")
else:
    print("Downloading the permitted DINOv2 / MPNet encoders.")
run_script("download_models.py", *model_args, log_name="models.log")
print(json.dumps(config, ensure_ascii=False, indent=2))


## 7. Training
Caches frame features, then trains for a fixed number of epochs. Don't add early stopping: with flipping,
order pairing stays near chance until about epoch 25.

The checkpoint is chosen by the clean rank score averaged with order pairing, and saved to Drive with the
negatives used. **If the session disconnects,** there's no resume, so start a new experiment from cell 2.
The frame cache stays in `colab_runs/frames` on Drive and is reused. The first full run spends extra time
creating it, including a smoke test.

In [ ]:
TRAINING_DIR = CHECKPOINT_DIR
FRAME_CACHE = DRIVE_STARTERKIT / "colab_runs/frames/features.pt"
OFFLINE = {"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1"}
SPLITS = DRIVE_RUN / "splits.json"
run_python(r"""
import json, sys
from pathlib import Path
from dataset import load_episodes, split_indices
cfg = json.load(open(sys.argv[1]))
episodes = load_episodes(cfg["data_dir"])
train, valid = split_indices(episodes, cfg["validation_fraction"], cfg["seed"])
Path(sys.argv[2]).write_text(json.dumps({"train": [episodes[i].episode_id for i in train],
                                         "valid": [episodes[i].episode_id for i in valid]}, indent=2))
""", CONFIG_PATH, SPLITS, log_name="splits.log")
run_python(r"""
import json, sys
from pathlib import Path
from common import load_config
from dataset import load_episodes
from team.negatives import load_action_labels
from team.rank_eval import load_candidates
config = load_config(sys.argv[1])
splits = json.loads(Path(sys.argv[2]).read_text())
episodes = {e.episode_id: e for e in load_episodes(config["data_dir"])}
valid = splits["valid"]
candidates = load_candidates(Path(config["output_dir"]).parent / "validation/candidates.json", valid,
                             [episodes[i].description for i in valid],
                             [load_action_labels(episodes[i]) for i in valid], config["seed"])
print("Validation episodes:", len(candidates))
""", CONFIG_PATH, SPLITS, log_name="candidates.log")
args = ["--config", CONFIG_PATH, "--splits", SPLITS, "--output-dir", TRAINING_DIR, "--frame-cache", FRAME_CACHE,
        "--model", MODEL, "--train-negatives", TRAIN_NEGATIVES, "--reverse-weight", REVERSE_WEIGHT,
        "--flip-probability", FLIP_PROBABILITY, "--save-negatives", "--device", "cuda:0"]
args += ["--epochs", 2, "--patience", 2, "--max-train-episodes", 64] if SMOKE_TEST else [
    "--epochs", EPOCHS, "--patience", EPOCHS]
run_command([PYTHON, "-u", "-m", "team.train_ranker", *args], log_name="train.log", extra_env=OFFLINE)
CHECKPOINT = TRAINING_DIR / "best.pt"
if not CHECKPOINT.is_file():
    raise FileNotFoundError("best.pt was not created. Check train.log.")
print("Selected checkpoint:", CHECKPOINT)


## 8. Scorecard
Scores from `report.json`. Clean rank score should be well above 0.375, order pairing above 0.5, and the
wrong-video scores on `similar_episode` and `lexical` near 0.375.

In [ ]:
report = json.loads((TRAINING_DIR / "report.json").read_text())
clean = (report["scores"]["similar_episode"]["score"] + report["scores"]["lexical"]["score"]) / 2
print(f"Best epoch: {report['best_epoch']}")
print(f"Clean rank score: {clean:.3f}")
print(f"Order pairing: {report['order_pairing']['score']:.3f}")
print("set | score | wrong video")
for name, item in report["scores"].items():
    print(f"{name} | {item['score']:.3f} | {report['wrong_video_scores'][name]['score']:.3f}")


## 9. Create the submission directory and check offline inference
Exports your trained MLP together with the encoders and tokenizer.
Using Train inputs, checks that every choice ID is returned exactly once and that repeated inference is consistent.
This is an interface check, and does not guarantee performance on the full competition data or passing the time limit.

In [ ]:
run_command([PYTHON, "-u", "-m", "team.export_ranker", "--checkpoint", CHECKPOINT,
             "--output-dir", EXPORT_DIR], log_name="export.log", extra_env=OFFLINE)
# Run from a separate CWD so that the exported modules are the ones actually used.
validator = LOCAL_ROOT / "validate-export.py"
validation_code = r"""
import sys
from pathlib import Path
artifact, kit, data = map(Path, sys.argv[1:4])
sys.path.insert(0, str(artifact))
sys.path.insert(1, str(kit))
import baseline_model, ranker_model
assert Path(baseline_model.__file__).parent == artifact
assert Path(ranker_model.__file__).parent == artifact
from validate import check_direct
check_direct(artifact, data, "cuda:0")
"""
validator.write_text(validation_code, encoding="utf-8")
run_command([PYTHON, "-I", validator, EXPORT_DIR, LOCAL_KIT, DATA_DIR],
            cwd=LOCAL_ROOT, log_name="validate.log",
            extra_env={**OFFLINE, "HF_HOME": str(LOCAL_ROOT / "empty-validation-cache")})


## 10. Create the ZIP and save to Drive
Packages with `submission.py` at the top level of the ZIP, and checks the size, file structure and checksum.
Submit only the generated ZIP to the competition site. Do not compress the whole `starterkit` folder.

Checks that the ZIP and its extracted size are each **at most 1 GiB** (as of the 2026-10-31 scoring environment).
The 256 MiB message shown by the existing `package.py` refers to an earlier server default.
For the final submission limits and competition rules, follow the latest guidance on the competition site.

In [ ]:
run_script("package.py", "--submission-dir", EXPORT_DIR, "--output", LOCAL_ZIP,
           "--force", log_name="package.log")
with zipfile.ZipFile(LOCAL_ZIP) as archive:
    names = archive.namelist()
    required_in_zip = {"submission.py", "submission.json", "weights/mlp.pt", "ranker_model.py",
                       "models/dinov2/model.safetensors", "models/mpnet/model.safetensors"}
    if not required_in_zip.issubset(names):
        raise ValueError("The submission ZIP's top-level structure is incorrect.")
    if json.loads(archive.read("submission.json")) != {"api_version": 2}:
        raise ValueError("Check api_version in submission.json.")
    expanded_bytes = sum(info.file_size for info in archive.infolist())
    if expanded_bytes > 1024**3 or LOCAL_ZIP.stat().st_size > 1024**3 or len(names) > 10000:
        raise ValueError("The submission ZIP exceeds the size or file count limit.")
    bad = archive.testzip()
    if bad:
        raise ValueError(f"ZIP integrity check failed: {bad}")

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(chunk)
    return digest.hexdigest()

local_sha256 = sha256_file(LOCAL_ZIP)
DRIVE_ZIP = DRIVE_RUN / LOCAL_ZIP.name
partial = DRIVE_ZIP.with_suffix(".zip.partial")
shutil.copyfile(LOCAL_ZIP, partial)
if sha256_file(partial) != local_sha256:
    raise IOError("The Drive copy's checksum differs. Rerun this cell.")
partial.replace(DRIVE_ZIP)
summary = {"run": RUN_NAME, "smoke_test": SMOKE_TEST,
           "checkpoint": str(CHECKPOINT), "submission_zip": str(DRIVE_ZIP),
           "zip_bytes": DRIVE_ZIP.stat().st_size, "expanded_bytes": expanded_bytes,
           "sha256": local_sha256, "files": len(names)}
(DRIVE_RUN / "submission-summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
if SMOKE_TEST:
    print("This ZIP is a small training result for functional checking. For full training, run a new experiment with SMOKE_TEST=False.")
print("Done! Download the ZIP above from Google Drive and submit it to the competition site.")
if DOWNLOAD_ZIP_TO_COMPUTER:
    from google.colab import files
    files.download(str(LOCAL_ZIP))


## Changing code and rerunning
Edit the code on Drive, then start a new experiment from cell 2. A copy of the code used for each run stays in its `source/` folder. If CUDA runs out of memory, lower the batch sizes in cell 2.